# Tutorial - AIME 2026: From LLMs to DNA

## 📚 Notebook 1. Training a Custom Genomic BERT from Scratch

Welcome to this comprehensive tutorial on building and adapting Genomic Language Models. In this notebook, we will walk through the entire pipeline of training a Transformer model from scratch on biological data and then evolving it using advanced transfer learning techniques.

### 🚀 What We Will Accomplish:

1.  **Data Acquisition & Engineering**:
    *   Load datasets from Hugging Face.
    *   Preprocess raw DNA sequences.

2.  **Custom Genomic Tokenization**:
    *   Construct a custom **Character-level** or **K-mer** tokenizer from scratch.
    *   Learn how to handle biological 'vocabulary' that lacks natural spaces.

3.  **From-Scratch Model Architecture**:
    *   Define a lightweight **'Mini-BERT'** configuration optimized for genomic tasks.
    *   Initialize the model weights and prepare for Masked Language Modeling (MLM).

4.  **The Training Cycle**:
    *   Implement a robust training loop with dynamic masking (15% probability).
    *   Visualize the learning profile using loss curves.

5.  **Transfer Learning & Continual Learning**:
    *   **Fine-Tuning**: Adapt our pre-trained human genomic model to Chromosome 22.
    *   **Continual Learning**: Introduce a completely new domain (viral genomics) while preserving foundational knowledge.

---



---
## 1. DATASET



### Step 1.1: Install Dependencies
First, we install the essential dependencies needed for genomic data processing, transformer training, visualization, and progress bars.

In [ ]:
!pip install datasets pandas transformers scikit-learn matplotlib tqdm openpyxl

### Step 1.2: Environment Initialization
We import our libraries

In [ ]:
import pandas as pd
from datasets import load_dataset
from datasets import load_from_disk
from tqdm import tqdm
import os

### Step 1.3: Load Dataset
### A Broad Overview of Genomic Datasets

In traditional genomics, datasets primarily consist of raw reference assemblies, functional annotations, and clinical variant databases. As genomic data science has scaled into the era of Deep Learning, these resources have become the bedrocks for training massive neural networks.

#### Core Repositories in Classical Genomics:
* **[NCBI GenBank](https://www.ncbi.nlm.nih.gov/genbank/)**: A comprehensive, publicly available database of nucleotide sequences and supporting bibliographic and biological annotations.
* **[Ensembl Genome Browser](https://www.ensembl.org/)**: A genome browser providing curated transcriptional frameworks, splice variants, and automated annotation tracks for vertebrate species.
* **[UCSC Genome Download Architecture](https://hgdownload.soe.ucsc.edu/)**: The primary storage layer for downloading massive biological data tracks, assembly builds (e.g., hg19, hg38), and conservation scores.

---

### Data Scaling for Genomic Foundation Models (gLMs)

Modern Genomic Language Models are trained on massive, multi-species corpuses containing billions or trillions of nucleotides. Instead of learning single-organism rules, they ingest sequence context across millions of diverse genomes to master evolutionary syntax.

#### State-of-the-Art Training Corpuses:
* **OpenGenome2 ([Evo2 Training Corpus](https://huggingface.co/datasets/arcinstitute/opengenome2))**: 8.8 trillion base pairs. All domains of life (Bacteria, Archaea, Eukaryota, Viruses). 5.52TB.
* **Nucleotide Transformer Dataset Collection ([InstaDeepAI](https://huggingface.co/InstaDeepAI))**: A collection of huaman and multi-species datasets.

> **More Genomic Datasets for gLMs:** **[Hugging Face Datasets](https://huggingface.co/datasets)**.

---

### Our Dataset for This Lab: `human-genome-cds`

**[`gonzalobenegas/human-genome-cds`](https://huggingface.co/datasets/gonzalobenegas/human-genome-cds)**

*   Human protein-coding sequences (CDS) extracted from the reference genome assembly.
*  CDS + 256 bp flanks
* train, test, val splits

In [ ]:
# Load Dataset
dataset = load_dataset("gonzalobenegas/human-genome-cds")

dataset

In [ ]:
# Create Dataframes
train = pd.DataFrame(dataset['train'])
test = pd.DataFrame(dataset['test'])
validation = pd.DataFrame(dataset['validation'])

train

In [ ]:
# Normalize lower case bases
train['seq'] = train['seq'].str.upper()
test['seq'] = test['seq'].str.upper()
validation['seq'] = validation['seq'].str.upper()

train

In [ ]:
# Chromosome and sequence counts
print(f"Total chromosomes in train: {train['chrom'].nunique()}")
print("Chromosome count in train:")
print(train['chrom'].value_counts().to_string())
print("-" * 40)

print(f"Total chromosomes in test: {test['chrom'].nunique()}")
print("Chromosome count in test:")
print(test['chrom'].value_counts().to_string())
print("-" * 40)

print(f"Total chromosomes in validation: {validation['chrom'].nunique()}")
print("Chromosome count in validation:")
print(validation['chrom'].value_counts().to_string())


print(train['seq'].str.len().describe())

print(test['seq'].str.len().describe())

print(validation['seq'].str.len().describe())

---
## 2. TOKENIZATION



### Step 2.1: Choose Your Tokenization Strategy

In this step, **you are the architect of your model's vocabulary.** Because DNA is a continuous sequence of chemical bases without natural spacing, we must decide how to break it into discrete pieces ("tokens") that our Transformer model can read.

Review the biological and computational trade-offs below before making your choice:

#### Option 1: Character-Level Tokenization (`"character"`)
* **How it works:** Every individual nucleotide base (`A`, `C`, `G`, `T`) is treated as an isolated single-letter word.
* **Biological Resolution:** High. The model observes every single point mutation directly.
* **Computational Demand:** High memory footprint. Because every base is an individual token, a 512-base-pair sequence creates 512 attention elements. Processing demands scale quadratically ($O(N^2)$) inside the model's attention layers.

#### Option 2: K-mer Fragment Tokenization (`"kmer"`)
* **How it works:** Nucleotides are grouped into overlapping multi-base words of length `K_SIZE`, sliding forward at intervals dictated by `STRIDE`.
* **Biological Resolution:** High structural context. Grouping bases allows the model to capture localized chemical motifs and reading frames (codons).
* **Computational Demand:** Highly efficient. Sliding the window forward with a `STRIDE > 1` substantially compresses sequence length, accelerating training on standard GPU hardware.

In [ ]:
# Choose your Tokenization strategy
TOKENIZATION_STYLE = "character"  # Options: "character" or "kmer"

# Only applies if TOKENIZATION_STYLE is "kmer"
K_SIZE = 6     # Length of each DNA "word"
STRIDE = 3      # Step size

# Directory path to cache the processed tokens on disk
SAVE_PATH = "./tokenized_human_genome_cds"

### Step 2.2: Initializing From-Scratch Tokenization Engines

Now, we initialize the backend machinery that transforms raw genetic strings into computer-readable formats. Because we are building this genomic model entirely from scratch, **both pipelines are completely DIY.** We do not borrow any pre-existing dictionaries or English language models.

Depending on the `TOKENIZATION_STYLE` you selected above, our codebase builds a native genomic vocabulary mapping framework:

1. **Track A: The Custom Character Strategy**
   This function builds a compact dictionary mapping standalone characters (`A, T, C, G, N`) directly to a unique integer index. Because it is manually designed, we explicitly inject the core Transformer control tokens (`[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`) right alongside our biological letters.

2. **Track B: The Custom K-mer Strategy**
   Instead of pulling a vocabulary from the internet, this pipeline uses mathematical permutations (`itertools.product`) to generate every single possible DNA combination of length `K_SIZE`. For a 6-mer, it automatically constructs all $4^6 = 4,096$ clean genomic words, tags on the 5 essential control tokens, and builds a pure, native biological tokenizer asset completely from scratch.

In [ ]:
import itertools
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import PreTrainedTokenizerFast

def character_tokenizer():
    """Builds an empty, custom vocabulary from scratch for individual bases."""
    vocab = {
        "[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4,
        "A": 5, "T": 6, "C": 7, "G": 8, "N": 9
    }
    base_tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]"))
    base_tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()

    return PreTrainedTokenizerFast(
        tokenizer_object=base_tokenizer,
        pad_token="[PAD]", unk_token="[UNK]",
        cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]"
    )

def build_custom_kmer_tokenizer(k):
    """Mathematically generates all unique DNA words and builds a pure custom tokenizer."""

    vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2, "[SEP]": 3, "[MASK]": 4}

    # Generate all possible permutations of A, T, C, G for the specified K-size
    bases = ['A', 'T', 'C', 'G']
    all_possible_kmers = [''.join(p) for p in itertools.product(bases, repeat=k)]

    # Map each generated k-mer string to a unique integer ID sequentially
    for i, kmer in enumerate(all_possible_kmers):
        vocab[kmer] = i + 5  # Offset by 5 to account for control tokens

    base_tokenizer = Tokenizer(models.WordLevel(vocab=vocab, unk_token="[UNK]"))
    base_tokenizer.pre_tokenizer = pre_tokenizers.WhitespaceSplit()

    return PreTrainedTokenizerFast(
        tokenizer_object=base_tokenizer,
        pad_token="[PAD]", unk_token="[UNK]",
        cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]"
    )

def kmer_string_splitter(seq, k, stride):
    """Acts as a text formatter that chops solid DNA into space-separated string sequences."""
    kmers = [seq[i:i+k] for i in range(0, len(seq) - k + 1, stride)]
    return " ".join(kmers)

print("✓ From-scratch genomic tokenization utilities compiled successfully.")

### Step 2.3: Dynamic Tokenizer Selection

The cell below reads your `TOKENIZATION_STYLE` parameter selection and dynamically constructs the active tokenizer instance.

In [ ]:
print(f"Allocating tokenizer asset for track selection: '{TOKENIZATION_STYLE.upper()}'\n")

if TOKENIZATION_STYLE == "character":
    # Initialize the manually built character tokenizer
    hf_tokenizer = character_tokenizer()
    print("✓ Success: Track A active. Custom character-level tokenizer built from scratch.")

elif TOKENIZATION_STYLE == "kmer":
    # Build a pure DNA vocabulary using mathematical permutations entirely from scratch
    hf_tokenizer = build_custom_kmer_tokenizer(K_SIZE)
    print(f"✓ Success: Track B active. Custom {K_SIZE}-mer tokenizer mathematically generated from scratch.")

else:
    raise ValueError(f"Invalid TOKENIZATION_STYLE option: '{TOKENIZATION_STYLE}'")

# Final verification printout
print(f"✓ Pipeline complete. Tokenizer active with a vocabulary of {hf_tokenizer.vocab_size:,} words.")

### Step 2.4: Execution

With our custom, purely biological tokenization engine built, we feed our data through it. We convert our raw Pandas DataFrames into lightweight Hugging Face `Dataset` structures and execute tokenization via `.map(..., batched=True)`.

By scaling our batch size to 10,000, we minimize Python loop overhead and allow the underlying tokenization architecture to process massive blocks of genetic data in parallel without risking a Google Colab crash.

In [ ]:
from datasets import Dataset

# Convert our clean Pandas DataFrames into optimized Hugging Face Datasets
print("Converting DataFrames to Hugging Face Dataset objects...")
hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
hf_val   = Dataset.from_pandas(validation)

# Define the isolated batch mapping function
def tokenization_batch_mapper(batch):
    """
    Processes micro-batches of data dynamically.
    Prevents memory bloat by vectorizing text sequentially.
    """
    processed_strings = []

    for raw_seq in batch['seq']:
        if TOKENIZATION_STYLE == "kmer":
            # kmer
            processed_strings.append(kmer_string_splitter(raw_seq, K_SIZE, STRIDE))
        else:
            # Character
            processed_strings.append(" ".join(list(raw_seq)))

    # Vectorize the current text batch into numerical tensor matrices using your custom vocab
    return hf_tokenizer(
        processed_strings,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

# Stream data through the batch engine
print("\nStarting batched tokenization...")

OPTIMIZED_BATCH_SIZE = 10000

tokenized_train = hf_train.map(tokenization_batch_mapper, batched=True, batch_size=OPTIMIZED_BATCH_SIZE, desc="Mapping Train")
tokenized_test  = hf_test.map(tokenization_batch_mapper, batched=True, batch_size=OPTIMIZED_BATCH_SIZE, desc="Mapping Test")
tokenized_val   = hf_val.map(tokenization_batch_mapper, batched=True, batch_size=OPTIMIZED_BATCH_SIZE, desc="Mapping Validation")

print("\n" + "="*50)
print("✓ SUCCESS: ALL DATA SPLITS SUCCESSFULLY TOKENIZED WITH CUSTOM VOCAB")
print("="*50)
print(f"Final Train Dataset Shape: {tokenized_train.shape}")
print(f"Available Tensor Features:  {list(tokenized_train.features.keys())}")
print("="*50)

### Step 2.5: Inspecting Tokenized Arrays

As a final verification check, we grab the first tokenized row and decrypt its numerical index matrix back into readable text. This allows us to confirm that our control structures (`[CLS]`, `[SEP]`, `[PAD]`) are aligned and ready for model consumption.

In [ ]:
# Extract the first processed sequence array
sample_input_ids = tokenized_train['input_ids'][0]

print("="*75)
print(f"             DECODING VERIFICATION ({TOKENIZATION_STYLE.upper()} TRACK)")
print("="*75)
print(f"Raw Numerical Input IDs (First 10 Tensors):\n{sample_input_ids[:10]}")
print(f"\nDecoded Text Output (First 10 Structural Words):\n{hf_tokenizer.decode(sample_input_ids[:10])}")
print("="*75)

Save tokenizer files in Colab

In [ ]:
import os

# Define a clean directory path for your custom tokenizer assets
tokenizer_export_dir = "custom_genomic_tokenizer"

print(f"Serializing custom '{TOKENIZATION_STYLE}' tokenizer setup to disk...")
hf_tokenizer.save_pretrained(tokenizer_export_dir)

print("\n" + "="*60)
print(f"✓ SUCCESS: TOKENIZER SAVED TO WORKSPACE DIRECTORY!")
print("="*60)
print(f"Target Location: ./{tokenizer_export_dir}/")
print(f"Files Written  : {os.listdir(tokenizer_export_dir)}")
print("="*60)
print("💡 Student Action: Download this folder or zip it to load it into Notebook 2!")

Zip and download tokenizer locally from Colab

In [ ]:
!zip -r /content/custom_genomic_tokenizer.zip /content/custom_genomic_tokenizer

In [ ]:
from google.colab import files
files.download('/content/custom_genomic_tokenizer.zip')

---
# 3. MODEL ARCHITECTURE DEFINITION





### Step 3.1: Define the Lightweight "Mini-BERT" Configuration
We initialize a compact, custom Transformer structural block. Because we are training a model completely from scratch in a limited time window, we intentionally configure a highly optimized "Mini-BERT" (2 hidden layers, 128 hidden size).

By passing `vocab_size=hf_tokenizer.vocab_size`, the model dynamically scales its embedding layer to fit your custom-built DNA dictionary perfectly.

In [ ]:
import torch
from transformers import BertConfig, BertForMaskedLM

# Dynamically target the runtime accelerator hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target execution hardware detected: {str(device).upper()}")

# Establish structural matrix constraints based on your token track
config = BertConfig(
    vocab_size=hf_tokenizer.vocab_size,  # Adapts instantly to character or k-mer vocab size
    hidden_size=128,                     # Internal representation embedding dimension size
    num_hidden_layers=2,                 # Light layer depth for highly responsive training
    num_attention_heads=2,               # Parallel self-attention processing heads
    intermediate_size=512,               # Feed-forward hidden expansion matrix scaling size
    max_position_embeddings=512          # Positional coordinate input boundary ceiling
)



### Step 3.2: Initialize the Network Object
We compile the final Masked Language Model object weights layout.

In [ ]:
print("Initializing BERT...")
model = BertForMaskedLM(config)

# Copy the entire model structure directly into the GPU
model.to(device)

print(f"\n✓ SUCCESS: Model successfully instantiated and mounted onto: {str(device).upper()}")
print(f"Total parameters tracking inside network: {sum(p.numel() for p in model.parameters()):,}")

# 4. MODEL TRAINING



### Step 4.1: DataLoader Stream Initialization
We pack our custom tokenized datasets into iterable pipelines called PyTorch `DataLoaders`. Inside these pipelines, our `DataCollatorForLanguageModeling` handles token manipulation on the fly.

Every time a batch of 32 sequences passes through, the collator randomly masks **15%** of the tokens. This forces the model to learn the fundamental grammar rules of the human genome by predicting the missing genetic bases.

We use standard **AdamW** weight decay optimization and prepare empty history tracking arrays to plot our performance trends later.

In [ ]:
import torch
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

# Drop text columns to prevent tensor errors
metadata_columns = ['chrom', 'start', 'end', 'strand', 'seq']
columns_to_drop = [col for col in metadata_columns if col in tokenized_train.column_names]

clean_train = tokenized_train.remove_columns(columns_to_drop)
clean_val   = tokenized_val.remove_columns(columns_to_drop)


# Sub-sampling
TRAIN_SUBSAMPLE_SIZE = 10240  # Generates exactly 40 batches per epoch at batch_size=256
VAL_SUBSAMPLE_SIZE   = 2048   # Generates exactly 8 validation batches per epoch

print(f"Filtering columns and downsampling splits for fast 1-minute run...")
subsampled_train = clean_train.select(range(TRAIN_SUBSAMPLE_SIZE))
subsampled_val   = clean_val.select(range(VAL_SUBSAMPLE_SIZE))


# Data collation
collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Data loaders
TRAIN_BATCH_SIZE = 256

print(f"Assembling PyTorch streaming pipelines with Batch Size: {TRAIN_BATCH_SIZE}...")

train_loader = DataLoader(
    subsampled_train,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    subsampled_val,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)


# Hyperparameter config
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
epochs = 10

# Clear tracking registers for a fresh run
train_loss_history = []
val_loss_history = []

print("\n" + "="*50)
print("   ✓ SUCCESS: FAST LIVE-DEMO DATA INFRASTRUCTURE STAGED!")
print("="*50)
print(f"Batches per Epoch (Steps) : {len(train_loader)}")
print(f"Total Combined Step Count : {len(train_loader) * epochs} optimization steps")
print("="*50)

### Step 4.2: Execute the Training Optimization Cycle
We execute the main training and validation loop. During the training phase, errors are calculated via cross-entropy loss, and gradients are propagated backwards (`loss.backward()`) to optimize the weights. During the validation phase, gradients are deactivated (`torch.no_grad()`) to quickly score the model's generalization capabilities on unseen genomic sequences.

In [ ]:
from tqdm.auto import tqdm

print("Starting training...")

for epoch in range(1, epochs + 1):
    # Training layer
    model.train()
    total_train_loss = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [TRAIN]")

    for batch in train_bar:
        optimizer.zero_grad()

        # Route processing tensors safely into GPU VRAM allocations
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Execute forward calculation pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backpropagate error structures and adjust structural weights
        loss.backward()
        optimizer.step()

        # Append step-level batch loss directly to history
        train_loss_history.append(loss.item())
        train_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    # Validation
    model.eval()
    total_val_loss = 0

    # Freeze internal gradients to conserve execution memory and accelerate performance
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [VALIDATION]"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

    # Calculate and record the average validation score for this epoch
    avg_val_loss = total_val_loss / len(val_loader)
    val_loss_history.append(avg_val_loss)

    print(f"\n» Epoch {epoch} Complete -> Step-End Train Loss: {train_loss_history[-1]:.4f} | Avg Val Loss: {avg_val_loss:.4f}\n")

### Step 4.3: Plot Train-Validation Loss Curve
We extract our recorded epoch histories and visualize them using a Matplotlib distribution chart. This allows to track optimization steps and verify that our validation loss tracks downward alongside our training loss curve, confirming the model is learning without overfitting.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

total_steps = len(train_loss_history)
num_epochs = len(val_loss_history)
steps_per_epoch = total_steps // num_epochs

loss_series = pd.Series(train_loss_history)
smoothed_loss = loss_series.rolling(window=5, min_periods=1).mean()

val_x_coords = [steps_per_epoch * i for i in range(1, num_epochs + 1)]

plt.figure(figsize=(10, 5))

plt.plot(
    range(1, total_steps + 1),
    train_loss_history,
    linestyle='-',
    color='#4A90E2',
    alpha=0.15,
    label='Raw Batch Loss (Dynamic Masking Noise)'
)

plt.plot(
    range(1, total_steps + 1),
    smoothed_loss,
    linestyle='-',
    linewidth=2,
    color='#1A5276',
    label='Smoothed Training Trend (Rolling Average)'
)

plt.plot(
    val_x_coords,
    val_loss_history,
    marker='s',
    markersize=8,
    linewidth=2.5,
    color='#E056FD',
    label='Validation Loss Curve (Per Epoch)'
)

plt.title("Genomic BERT Training Profile: Batch-Level Training vs Epoch-Level Validation", fontsize=12, fontweight='bold')
plt.xlabel("Total Optimization Steps (Batches Processed)")
plt.ylabel("Cross-Entropy Loss Magnitude")

for x in val_x_coords:
    plt.axvline(x=x, color='gray', linestyle=':', alpha=0.4)

plt.grid(True, alpha=0.1)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Step 4.4: Save Weights Checkpoint
Finally, we save the trained model weights matrix (`state_dict`) as a raw checkpoint binary onto disk for storage or future downstream tasks.

In [ ]:
checkpoint_name = "genomic_bert_weights.pt"
torch.save(model.state_dict(), checkpoint_name)

print("\n" + "="*75)
print(f"✓ CHECKPOINT SUCCESS: Matrix successfully written as: '{checkpoint_name}'")
print("="*75)

# 5. TRANSFER LEARNING PARADIGMS



**Transfer learning** is a machine learning technique where a model developed for one task is reused as the starting point for a second, related task.

Instead of training a model from scratch (which requires massive datasets and computing power) you leverage the "knowledge" a model has already gained from a previous, massive dataset and apply it to a new problem.

### Why It Matters
* **Saves Time & Money:** Reduces training time from days to minutes.
* **Requires Less Data:** Works effectively even if you only have a few hundred examples for your new task.
* **Better Performance:** Starting with a foundational understanding leads to higher accuracy than building from scratch.



## 5.1 Fine-Tuning

**Fine-tuning** is a specific approach to transfer learning where you take a model that has already been trained on a massive, general dataset (a **pre-trained model**) and perform additional training on a smaller, specialized dataset to adapt it to a specific task.

Instead of training a model from scratch, you use the pre-trained model as a highly advanced head start.



---

### How It Works

1. **Start with a Pre-trained Base:** You take a "foundation model" that already understands general patterns. For example, a language model trained on the entire internet, or a genomic model trained on all known human DNA.
2. **Add a Task-Specific Layer:** You typically replace or modify the very last layer of the network (the "output head") so it matches your specific goal (e.g., changing a next-token predictor into a tumor/benign classifier).
3. **Train on Target Data:** You run a small amount of specialized data through the network using a very low learning rate. This gently nudges the model's existing internal weights so it optimizes for your new task without erasing its foundational knowledge.



---

### Chromosome Fine-Tuning

We are going to take our pretrained genomic **MiniBERT** model and fine-tune it specifically on **Chromosome 22**.

Up to this point, the model has been trained on all other chromosomes *except* Chromosome 22. By freezing its foundational understanding of genetic "grammar" and training it on this remaining dataset, we are adapting the model to become an expert on the unique sequence motifs, structural features, and functional elements specific to Chromosome 22.


In [ ]:
import torch
from transformers import BertForMaskedLM, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Load the test set and prepare for tokenization
hf_test_dataset = dataset['test']

# Re-apply tokenization to the test set
print("Tokenizing test set for fine-tuning...")
tokenized_ft_test = hf_test_dataset.map(tokenization_batch_mapper, batched=True, batch_size=10000)

# Clean text columns
columns_to_drop = ['chrom', 'start', 'end', 'strand', 'seq']
ft_test_clean = tokenized_ft_test.remove_columns([c for c in columns_to_drop if c in tokenized_ft_test.column_names])

# Load the model architecture and saved weights
print("Loading model from checkpoint: genomic_bert_weights.pt")
ft_model = BertForMaskedLM(config)
ft_model.load_state_dict(torch.load("genomic_bert_weights.pt"))
ft_model.to(device)

# Prepare DataLoader
collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=True, mlm_probability=0.15)
ft_loader = DataLoader(ft_test_clean, batch_size=32, shuffle=True, collate_fn=collator)



In [ ]:
# Fine-tuning Setup
optimizer = torch.optim.AdamW(ft_model.parameters(), lr=5e-5)
ft_epochs = 3
ft_model.train()

print(f"Starting fine-tuning for {ft_epochs} epochs...")
for epoch in range(1, ft_epochs + 1):
    total_loss = 0
    loop = tqdm(ft_loader, desc=f"FT Epoch {epoch}/{ft_epochs}")
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = ft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(ft_loader)
    print(f"Epoch {epoch} Fine-Tuning Loss: {avg_loss:.4f}")

print("\nFine-tuning complete!")

## 5.2 Continual Learning

**Continual Learning** is a training paradigm where a model learns from new data over time, acquiring new knowledge while retaining what it has already learned.

Unlike standard pretraining, where a model is trained once on a static dataset and never changes, continual learning allows a model to adapt to new tasks, environments, or distributions.


### Comparison: Fine-Tuning vs. Continual Learning

| Feature | Fine-Tuning | Continual Learning |
| :--- | :--- | :--- |
| **Objective** | Adapt a model to *one* specific target task. | Adapt a model to a *sequence* of multiple tasks over time. |
| **Old Tasks** | It does not matter if the model forgets the original pre-training data. | The model *must* remember how to solve previous tasks. |
| **Data Stream** | One-time specialized dataset. | An ongoing, non-stationary stream of data. |



---

## Continual Learning on SARS-CoV-2

Now, we are going to take our human chromosome  model and perform **continual learning** using a SARS-CoV-2 dataset.

Instead of freezing the model or training it on a narrow, isolated task, we are introducing a completely new biological domain: viral genomics. The model will continuously update its weights to adapt to the highly compact, non-coding-sparse nature of the viral kingdom.

The core challenge here is balancing **plasticity and stability**: while the model updates its parameters to master the specific signatures of SARS-CoV-2, it must do so without suffering from *catastrophic forgetting*. It needs to retain the foundational, long-range structural rules of genomic data it originally learned from the human genome.

In [ ]:
!wget https://raw.githubusercontent.com/pabloarozarena/aime2026-T7-genomic-llms/main/data/sarscov2.csv.gz -O sarscov2.csv.gz
!gunzip -f sarscov2.csv.gz

In [ ]:
import pandas as pd
df = pd.read_csv("sarscov2.csv")
df["sequence"] = df["sequence"].astype(str)
pd.DataFrame(df)

In [ ]:
# Data splits: Training on Pre-Omicron Data
from sklearn.model_selection import train_test_split

test_df = df[df["Variant"] == "Omicron"].reset_index(drop=True)
non_omicron_df = df[df["Variant"] != "Omicron"].reset_index(drop=True)

train_df, val_df = train_test_split(
    non_omicron_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test Omicron:", test_df.shape)


In [ ]:
# Tokenization and Continual Learning
import torch
from datasets import Dataset
from transformers import BertForMaskedLM, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

def tokenize_sarscov2(batch):
    if TOKENIZATION_STYLE == "kmer":
        texts = [
            kmer_string_splitter(seq.upper(), K_SIZE, STRIDE)
            for seq in batch["sequence"]
        ]
    else:
        texts = [
            " ".join(list(seq.upper()))
            for seq in batch["sequence"]
        ]

    return hf_tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize_sarscov2, batched=True)
val_ds = Dataset.from_pandas(val_df, preserve_index=False).map(tokenize_sarscov2, batched=True)
test_ds = Dataset.from_pandas(test_df, preserve_index=False).map(tokenize_sarscov2, batched=True)

train_ds = train_ds.remove_columns(["Variant", "sequence"])
val_ds = val_ds.remove_columns(["Variant", "sequence"])
test_ds = test_ds.remove_columns(["Variant", "sequence"])

collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=True,
    mlm_probability=0.15
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collator)

cl_model = BertForMaskedLM(config)
cl_model.load_state_dict(torch.load("genomic_bert_weights.pt", map_location=device))
cl_model.to(device)

optimizer = torch.optim.AdamW(cl_model.parameters(), lr=5e-5)
cl_epochs = 3

for epoch in range(1, cl_epochs + 1):
    cl_model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{cl_epochs}"):
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}
        loss = cl_model(**batch).loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    cl_model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            val_loss += cl_model(**batch).loss.item()

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss / len(train_loader):.4f} | "
        f"Val loss: {val_loss / len(val_loader):.4f}"
    )

cl_model.eval()
test_loss = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        test_loss += cl_model(**batch).loss.item()

print(f"Omicron test loss: {test_loss / len(test_loader):.4f}")

torch.save(cl_model.state_dict(), "minibert_sarscov2_continual_learning.pt")


## 5.3 Alternative Transfer Learning Techniques



Beyond full fine-tuning and continual learning, there are several highly efficient strategies used to adapt foundation models to downstream tasks without rewriting all of the model's weights.

---

### 1. Feature Extraction (Frozen Backbone)
Instead of updating the entire network, you "freeze" all the layers of the pre-trained model and treat it as a fixed feature extractor.

* **How it works:** You pass your new dataset through the frozen model to get its internal representations (embeddings), and then train a lightweight, shallow classifier (like a linear layer, SVM, or Random Forest) on top of those outputs.
* **Best used for:** Situations with extremely limited target data or minimal compute resources, as it completely avoids calculating gradients for the massive base model.

---

### 2. Parameter-Efficient Fine-Tuning (PEFT)
PEFT methods allow you to adapt giant models by training only a tiny fraction (often $< 1\%$) of the total parameters, keeping the original weights untouched.

* **LoRA (Low-Rank Adaptation):** Instead of modifying the massive weight matrices directly, LoRA injects small, trainable rank-decomposition matrices alongside the original layers. This drastically reduces GPU memory requirements during training.
* **Prefix Tuning / Prompt Tuning:** You append continuous, trainable virtual tokens (embeddings) to the input sequence. Only these prompt-specific parameters are updated during training, effectively "steering" the frozen model to the correct task.



---

### 3. Domain Adaptation
This technique is used when the source task and target task are identical, but the underlying data distribution changes (a phenomenon known as "domain shift").

* **How it works:** If you have a genomic model trained on high-quality, lab-verified reference genomes, but you want to deploy it on noisy, low-coverage sequencing data, you use domain adaptation. Algorithms align the feature spaces of both domains so the model learns features that are invariant to the noise.
* **Best used for:** Bridging the gap between simulation vs. reality, or distinct experimental setups.

---

### 4. Zero-Shot & Few-Shot Learning (In-Context Learning)
Popularized by modern LLMs and large foundation models, this technique requires **zero** weight updates or traditional training.

* **How it works:** You leverage the existing emergent capabilities of the model by providing it with a descriptive instruction (**Zero-Shot**) or a few input-output examples directly inside the context window (**Few-Shot**). The model deduces the task on the fly during the forward pass.
* **Best used for:** Rapid prototyping, interactive workflows, and situations where updating model weights is completely restricted.

##